In [ ]:
import numpy as np


class SGD:
    def __init__(self, learning_rate=0.01, epochs=100, random_state=42):
        self.lr = learning_rate
        self.epochs = epochs
        self.random_state = random_state
        self.w = None
        self.b = 0.0
        self.loss_history = []

    def fit(self, x, y):
        x = np.asarray(x, dtype=float)
        y = np.asarray(y, dtype=float).reshape(-1)

        if x.ndim != 2:
            raise ValueError("the dimensions are less than 2")
        if len(x) != len(y) or len(y) == 0:
            raise ValueError("some values are missing")
        if not np.isfinite(x).all() or not np.isfinite(y).all():
            raise ValueError("its going towards infinity")

        n_samples, n_features = x.shape

        self.w = np.zeros(n_features)
        self.b = 0.0
        self.loss_history = []

        rng = np.random.default_rng(self.random_state)

        for epochs in range(self.epochs):
            indices = rng.permutation(n_samples)

            for i in indices:
                xi = x[i]
                yi = y[i]

                y_pred = xi @ self.w + self.b
                error = y_pred - yi

                dw = 2 * xi * error
                db = 2 * error

                self.w = self.w - self.lr * dw
                self.b = self.b - self.lr * db

            ep_pred = x @ self.w + self.b
            epochs_loss = np.mean((ep_pred - y) ** 2)
            self.loss_history.append(epochs_loss)

        return self

    def predict(self, x):
        if self.w is None:
            raise ValueError("Fit the model before predicting.")

        x = np.asarray(x, dtype=float)

        if x.ndim != 2:
            raise ValueError("X must be a 2D array.")

        return x @ self.w + self.b

In [ ]:
rng = np.random.default_rng(42)

X = np.arange(1, 101, dtype=float).reshape(-1, 1)
y = 4 * X[:, 0] + 3 + rng.normal(0, 2, size=len(X))

# Scale the feature to make optimization easier
X_mean = X.mean(axis=0)
X_std = X.std(axis=0)
X_scaled = (X - X_mean) / X_std

model = SGD(learning_rate=0.001, epochs=100, random_state=42)

model.fit(X_scaled, y)

# Convert learned coefficient back to original x scale
original_weight = model.w[0] / X_std[0]
original_bias = model.b - original_weight * X_mean[0]

print("Learned weight:", original_weight)
print("Learned bias:", original_bias)
print("Final MSE:", model.loss_history[-1])

print(
    "Predictions:", model.predict((np.array([[1.0], [5.0], [10.0]]) - X_mean) / X_std)
)

Learned weight: 3.9916626869867287
Learned bias: 3.3186515872428686
Final MSE: 2.3289469455773997
Predictions: [ 7.31031427 23.27696502 43.23527846]


In [ ]:
import numpy as np
from sklearn.datasets import fetch_california_housing
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LinearRegression, SGDRegressor
from sklearn.pipeline import make_pipeline
from sklearn.metrics import r2_score, mean_squared_error

# Load data
housing = fetch_california_housing(as_frame=True)
X = housing.data.to_numpy()
y = housing.target.to_numpy()

# Split
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

# Scale features
x_scaler = StandardScaler()
X_train_scaled = x_scaler.fit_transform(X_train)
X_test_scaled = x_scaler.transform(X_test)

# Scale target (fit only on training target)
y_scaler = StandardScaler()
y_train_scaled = y_scaler.fit_transform(y_train.reshape(-1, 1)).ravel()

# Pipeline 1: Linear Regression baseline
pipeline1 = make_pipeline(StandardScaler(), LinearRegression())
pipeline1.fit(X_train, y_train)
y_pred1 = pipeline1.predict(X_test)

# Pipeline 2: Your scratch SGD class
# Assumes SGD supports multiple features and fit(X, y).
pipeline2 = SGD(learning_rate=0.001, epochs=100)
pipeline2.fit(X_train_scaled, y_train_scaled)

pred2_scaled = pipeline2.predict(X_test_scaled)
y_pred2 = y_scaler.inverse_transform(np.asarray(pred2_scaled).reshape(-1, 1)).ravel()

# Pipeline 3: sklearn SGDRegressor
pipeline3 = make_pipeline(
    StandardScaler(),
    SGDRegressor(
        loss="squared_error",
        penalty=None,
        learning_rate="constant",
        eta0=0.001,
        max_iter=1000,
        tol=1e-4,
        shuffle=True,
        random_state=42,
    ),
)

# Fit on scaled target, then convert predictions back
pipeline3.fit(X_train, y_train_scaled)
pred3_scaled = pipeline3.predict(X_test)
y_pred3 = y_scaler.inverse_transform(pred3_scaled.reshape(-1, 1)).ravel()

# Evaluate
for name, pred in [
    ("Linear Regression", y_pred1),
    ("Scratch SGD", y_pred2),
    ("sklearn SGDRegressor", y_pred3),
]:
    print(f"\n{name}")
    print("R²:", round(r2_score(y_test, pred), 4))
    print("RMSE:", round(np.sqrt(mean_squared_error(y_test, pred)), 4))

C:\Users\mahad\AppData\Local\Temp\ipykernel_7016\1908598757.py:48: RuntimeWarning: overflow encountered in square
  epochs_loss=np.mean((ep_pred-y)**2)
c:\Users\mahad\AppData\Local\Programs\Python\Python313\Lib\site-packages\numpy\_core\_methods.py:132: RuntimeWarning: overflow encountered in reduce
  ret = umr_sum(arr, axis, dtype, out, keepdims, where=where)



Linear Regression
R²: 0.5758
RMSE: 0.7456

Scratch SGD
R²: -inf
RMSE: inf

sklearn SGDRegressor
R²: -304659.6011
RMSE: 631.8465


c:\Users\mahad\AppData\Local\Programs\Python\Python313\Lib\site-packages\sklearn\metrics\_regression.py:1297: RuntimeWarning: overflow encountered in square
  numerator = xp.sum(weight * (y_true - y_pred) ** 2, axis=0)
c:\Users\mahad\AppData\Local\Programs\Python\Python313\Lib\site-packages\sklearn\metrics\_regression.py:584: RuntimeWarning: overflow encountered in square
  (y_true - y_pred) ** 2, axis=0, weights=sample_weight, xp=xp


In [ ]:
# i know i got worst case scenario with SGD and Stochasticgdregressor,